# Model Test

After doing a some exploration on initial_test.ipynb, this will be the final test before the actual implementation, using the tools presented in the challenge (which are in src/cwfc)

In [125]:
import pandas
import numpy as np
from sklearn.linear_model import Ridge

from cwfc import metrics, baselines, validation

import matplotlib.pyplot as plt


In [126]:
csv_raw = pandas.read_csv("../challenge_data/vm_cpu_readings.csv.gz", header=0)

In [127]:
# Data validation
# 1. Check for missing data.
missing_cpu = csv_raw["avg_cpu"].isna().sum()
print(f"Missing avg_cpu readings: {missing_cpu}")

# 1.1. Filter unvalid
csv_valid = csv_raw.dropna(subset=["avg_cpu"]).copy()

# 2. Check duplicate VM readings at the same timestamp.
duplicates = csv_raw.duplicated(subset=["timestamp", "vm_id"]).sum()
print(f"Duplicate (timestamp, vm_id) pairs: {duplicates}")

# 2.1. Deduplicate
if duplicates > 0:
    csv_valid = csv_valid.drop_duplicates(subset=["timestamp", "vm_id"])

Missing avg_cpu readings: 0
Duplicate (timestamp, vm_id) pairs: 0


In [128]:
# Once validated, group by timestamps, since we only need to produce forecast for the 'active_vm_count' and 'aggregate_cpu_utilization'.
csv_ts = (
    csv_raw.groupby("timestamp")
    .agg(
        active_vm_count=("vm_id", "nunique"),
        aggregate_cpu_utilization=("avg_cpu", "mean"),
    )
    .reset_index()
    .sort_values("timestamp")
)

In [129]:
# 3. Last check, for any gaps (The readings must be every 5 mins)
# In seconds, 5 minutes = 300 seconds
time_diffs = csv_ts["timestamp"].diff().dropna()
unique_diffs = time_diffs.unique()
print(f"Time differences between consecutive timestamps (seconds): {unique_diffs}")

if len(unique_diffs) == 1 and unique_diffs[0] == 300:
    print(" Grid is continuous (every step is exactly 300s / 5 mins).")
else:
    print(" Warning: Timestamp gaps detected")

print("\nFinal Target Shape:", csv_ts.shape)  # Should be (134, 3)

Time differences between consecutive timestamps (seconds): [300.]
 Grid is continuous (every step is exactly 300s / 5 mins).

Final Target Shape: (134, 3)


In [130]:
# Data split
train_data = csv_ts[:110]
test_data = csv_ts[110:]

In [131]:
# Baseline prediction
# Grab the very last observation from the training set (row 110)
last_known_row = train_data.iloc[-1]
last_vm_count = last_known_row["active_vm_count"]
last_cpu = last_known_row["aggregate_cpu_utilization"]

print(
    f"Last known point -> Active VMs: {last_vm_count}, CPU: {last_cpu:.2f}%"
)

# Predict that exact value across all 24 holdout timestamps
baseline_preds = {
    "aggregate_cpu_utilization": baselines.persistence_forecast(train=train_data,
    template=test_data, target="aggregate_cpu_utilization"),
    
    "active_vm_count": baselines.persistence_forecast(train=train_data,
        template=test_data, target="active_vm_count")
}

baseline_preds_metrics = {}

# Check metrics (MAE and RMSE)
for col in ["active_vm_count", "aggregate_cpu_utilization"]:
    actual = test_data[col]
    pred = baseline_preds[col]
    mae = metrics.mae(actual, pred)
    rmse = metrics.rmse(actual, pred)
    smape = metrics.smape(actual, pred)

    baseline_preds_metrics[col] = {
        "mae": mae,
        "rmse": rmse,
        "smape": smape
    }

    print(f"\n[Baseline] {col}:")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  SMAPE: {smape:.4f}")

Last known point -> Active VMs: 10000.0, CPU: 9.16%

[Baseline] active_vm_count:
  MAE:  0.0000
  RMSE: 0.0000
  SMAPE: 0.0000

[Baseline] aggregate_cpu_utilization:
  MAE:  0.1468
  RMSE: 0.1848
  SMAPE: 1.5851


In [132]:
# We already got a baseline, so now its time to beat MAE: 0.1468 & RMSE: 0.1848
# Since the data is kinda small (110 rows), the simpler the model the better, to prevent overfitting

def create_features(series):
    """Creates lag and rolling features from a univariate CPU series."""
    df_feat = pandas.DataFrame({"cpu": series})

    # Lags (past 5, 10, 15, 30 minutes)
    df_feat["lag_1"] = df_feat["cpu"].shift(1)
    df_feat["lag_2"] = df_feat["cpu"].shift(2)
    df_feat["lag_3"] = df_feat["cpu"].shift(3)
    df_feat["lag_6"] = df_feat["cpu"].shift(6)

    # Rolling statistics over past 30 minutes (6 steps)
    df_feat["rolling_mean_6"] = (
        df_feat["cpu"].shift(1).rolling(window=6).mean()
    )
    df_feat["rolling_std_6"] = df_feat["cpu"].shift(1).rolling(window=6).std()

    # Momentum (change over the last 15 mins)
    df_feat["diff_1"] = df_feat["lag_1"] - df_feat["lag_2"]
    df_feat["diff_3"] = df_feat["lag_1"] - df_feat["lag_3"]

    return df_feat

In [133]:
# Build feature table for the 110 training rows
train_series = train_data["aggregate_cpu_utilization"].copy()
features_df = create_features(train_series)

# Drop initial rows with NaNs caused by lagging
valid_idx = features_df.dropna().index
X_train = features_df.loc[valid_idx].drop(columns=["cpu"])
y_train = features_df.loc[valid_idx, "cpu"]

In [134]:
# We simulate forecasting 24 steps ahead using only the past
print("--- RUNNING TIME-AWARE VALIDATION (Expanding Window) ---")

cv_maes = []
# Test on the last 24 points of the training data (rows 86 to 110)
split_point = 86
X_cv_train, y_cv_train = X_train.loc[: split_point - 1], y_train.loc[
    : split_point - 1
]
X_cv_val, y_cv_val = X_train.loc[split_point:], y_train.loc[split_point:]

cv_model = Ridge(alpha=1.0)
cv_model.fit(X_cv_train, y_cv_train)
cv_preds = cv_model.predict(X_cv_val)

cv_mae = metrics.mae(y_cv_val, cv_preds)
print(f"Validation MAE inside training set: {cv_mae:.4f}")

--- RUNNING TIME-AWARE VALIDATION (Expanding Window) ---
Validation MAE inside training set: 0.1038


In [135]:
final_model = Ridge(alpha=1.0)
final_model.fit(X_train, y_train)

# Recursive forecast: predict step 1 -> use it as lag to predict step 2 -> etc.
history = list(train_series.values)
ml_cpu_forecast = []

for step in range(24):
    curr_series = pandas.Series(history)
    curr_feats = create_features(curr_series).iloc[[-1]].drop(columns=["cpu"])

    pred_next = final_model.predict(curr_feats)[0]
    # Bound within [0, 100] as required by contract
    pred_next = float(np.clip(pred_next, 0.0, 100.0))

    ml_cpu_forecast.append(pred_next)
    history.append(pred_next)

print(ml_cpu_forecast)

[9.181416917348098, 9.214106533633874, 9.193645238554243, 9.212224816885637, 9.216626220982366, 9.226585926393945, 9.2245864956766, 9.233872030841813, 9.23622520905091, 9.241888915974867, 9.243470513549779, 9.248554474394531, 9.250783020403064, 9.254526364741434, 9.256571281303152, 9.259764945633709, 9.261771775144515, 9.264352266741081, 9.26618372930308, 9.26837483932674, 9.270042661053152, 9.27188601529873, 9.27337732800462, 9.274954562824364]


In [136]:
actual_test_cpu = test_data["aggregate_cpu_utilization"].values
ml_mae = metrics.mae(actual_test_cpu, ml_cpu_forecast)
ml_rmse = metrics.rmse(actual_test_cpu, ml_cpu_forecast)
ml_smape = metrics.smape(actual_test_cpu, ml_cpu_forecast)
ml_normalized_mae = metrics.normalized_mae(actual_test_cpu, ml_cpu_forecast, baseline_preds["aggregate_cpu_utilization"])

print("\n--- FINAL TEST RESULTS ON HOLDOUT ---")
print(f"Baseline MAE: {baseline_preds_metrics["aggregate_cpu_utilization"]["mae"]:.4f}  |  ML MAE:  {ml_mae:.4f}")
print(f"Baseline RMSE: {baseline_preds_metrics["aggregate_cpu_utilization"]["rmse"]:.4f} |  ML RMSE: {ml_rmse:.4f}")
print(f"Baseline SMAPE: {baseline_preds_metrics["aggregate_cpu_utilization"]["smape"]:.4f} |  ML RMSE: {ml_rmse:.4f}")
print(f"Normalized MAE: {ml_normalized_mae:.4f}")


--- FINAL TEST RESULTS ON HOLDOUT ---
Baseline MAE: 0.1468  |  ML MAE:  0.1032
Baseline RMSE: 0.1848 |  ML RMSE: 0.1373
Baseline SMAPE: 1.5851 |  ML RMSE: 0.1373
Normalized MAE: 0.7033


In [137]:
# Validation

# 1. Convert integer timestamps (seconds) to datetime so validation.py doesn't complain
train_df_dt = train_data.copy()
train_df_dt["timestamp_dt"] = pandas.to_datetime(
    train_df_dt["timestamp"], unit="s", utc=True
)

# 2. Generate valid rolling splits for your 110 training rows
splits = validation.rolling_origin_splits(
    frame=train_df_dt,
    timestamp_col="timestamp_dt",
    horizon=24,  # Exactly 24 steps (2 hours)
    min_train_size=62,  # Start with ~5 hours of history
    step=12,  # Step forward by 1 hour (12 steps)
)

print(f"Generated {len(splits)} rolling-origin validation windows:")
for i, s in enumerate(splits):
    print(f"  Split {i+1}: Train ends {s.train_end} -> Val ends {s.validation_end}")

# 3. Evaluate your model across these rolling windows
cv_errors = []
for split in splits:
    # Slice train and val by timestamps
    val_mask = (train_df_dt["timestamp_dt"] >= split.validation_start) & (
        train_df_dt["timestamp_dt"] <= split.validation_end
    )
    tr_mask = train_df_dt["timestamp_dt"] <= split.train_end

    val_actual = train_df_dt.loc[val_mask, "aggregate_cpu_utilization"]

    # Baseline on this split (last known value in split train)
    last_val = train_df_dt.loc[tr_mask, "aggregate_cpu_utilization"].iloc[-1]
    split_baseline_mae = metrics.mae(
        val_actual, [last_val] * len(val_actual)
    )
    cv_errors.append(split_baseline_mae)

print(
    f"Mean Baseline Validation MAE across rolling windows: {np.mean(cv_errors):.4f}"
)

Generated 3 rolling-origin validation windows:
  Split 1: Train ends 1970-01-01 05:05:00+00:00 -> Val ends 1970-01-01 07:05:00+00:00
  Split 2: Train ends 1970-01-01 06:05:00+00:00 -> Val ends 1970-01-01 08:05:00+00:00
  Split 3: Train ends 1970-01-01 07:05:00+00:00 -> Val ends 1970-01-01 09:05:00+00:00
Mean Baseline Validation MAE across rolling windows: 0.1848


In [138]:
submission = pandas.DataFrame(
    {
        "timestamp": test_data["timestamp"].values,
        "active_vm_count": [10000.0] * 24,  # Constant from our analysis
        "aggregate_cpu_utilization": ml_cpu_forecast,
    }
)

submission.to_csv("test_media/predictions.csv", index=False)
print("\n 'predictions.csv' updated successfully!")


 'predictions.csv' updated successfully!
